# Reflexion: 自然语言强化学习

基于梯度的强化学习需要成千上万次尝试以及一个CPU集群来修复一种失败模式。Reflexion 通过自然语言做到了：在每次失败尝试之后，agent写一份反思，保存在情景记忆中，通过这份记忆约束下一次尝试。这就是Letta‘s 的闲时计算、Claude Code的CLAUDE.md 学习，以及pro-workflow的学习规则。

## 问题描述

当一个agent在某项任务上失败。在标准的强化学习中，你需要额外跑几千次尝试、计算梯度、更新权重。贵、慢，而且大部分生产agents没有用于每种失败尝试的训练预算。

Reflexion 问了一个不同的问题：如果agent只是想一想为什么失败了，然后带着这个反思赛道提示词中再做一次呢？没有权重更新，没有梯度，只有不同尝试间存储的自然语言。

## 基本概念

### 三个组件

```
Actor:   生成一份轨迹（ReAct风格的循环）
Evaluator：  为这份轨迹打分————二分、启发式或者自我评价
Self-Reflector:  基于失败写一份自然语言反思
```
额外加上一份数据结构
```
情景记忆：先前的反思列表，前置到下一次尝试的提示词之前。
```
Actor 跑一次尝试，Evaluator对其进行打分。如果分数较低，Self-Reflector生成一份反思（“因为我读错了问题，将对Y的提问理解成了对X的提问，导致调用了错误的工具”）。下一次尝试从头开始，不过能够看见这次反思。

### 三种评估类型
- 标量 ————外部的二进制信号。 成功了还是失败了、通过了还是失败了，简单但最强的信号。
- 启发式 ————预设的失败签名。 “如果agent面对同一个问题产生了两次相同的行动，标记为阻塞”。 “如果轨迹超过了50步，标记为低效”。
- 自我评估 ————LLM对轨迹进行打分。当没有事实依据时可用。信号较弱

2026年默认是混合，有标量用标量，否则自我评估，启发式作为安全轨迹。

### 为什么泛化了

Reflexion 不是一种新的算法，基本上所有“自我修复”的agent都在跑它的某种变体。
- Letta‘s 的 sleep-time compute。 一个单独的agent对过往的会话做反思，然后写到记忆块中。
- Claude Code的`CLAUDE.md`/“save memory”模式。 反思作为学到的东西，前缀注入未来的会话。
- pro-workflow的`learn-rule`命令。 纠正被捕获为显式的规则。
- LangGraph的反思节点。 一个对输出打分，在必要时路由到修订的节点。

从相同的洞见派生：自然语言是一个足够丰富的媒介，能够在多次运行之间承载“我从失败中学到了什么”。

### 什么时候有效、什么时候无效

有效
- 有明确的失败信号（测试失败、工具错误、回答错误）
- 任务类型可重复（重复问相同的问题）
- 反思有改进轨迹的余地

无效
- 第一次尝试时就成功了
- 失败模式时来自外部的； 比如“网络波动失败”对下一次运行没有什么用处。
- 反思变成了迷信，给一次偶发的不稳定运行存下了一套叙事。

2026年的坑，记忆腐烂。反思越攒越多，有些已经过时或错误。随着情景记忆缓存增长，重试变慢。

# 开始编码

本章核心：**Actor → Evaluator → Self-Reflector → 情景记忆**。  
先用脚本化玩具看清重试循环；再用 **PyTorch 小评估器** 示意可学习打分；最后用 **LangGraph + DeepSeek** 做生产 Reflexion。


## 1. 教学玩具：Reflexion 循环骨架


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable

from typing_extensions import TypedDict


class EvalKind(str, Enum):
    """三种评估类型（对应笔记）。"""

    SCALAR = "scalar"
    HEURISTIC = "heuristic"
    SELF = "self"


@dataclass
class ToolCall:
    """一次工具调用。"""

    name: str
    args: dict[str, Any]


@dataclass
class Step:
    """轨迹中的一步。"""

    thought: str
    action: str
    args: dict[str, Any]
    observation: str


@dataclass
class Trajectory:
    """Actor 一次尝试的完整轨迹。"""

    question: str
    steps: list[Step] = field(default_factory=list)
    final_answer: str = ""


@dataclass
class EvalResult:
    """Evaluator 输出。"""

    kind: EvalKind
    success: bool
    score: float
    reason: str


@dataclass
class Reflection:
    """Self-Reflector 产出的自然语言反思。"""

    attempt: int
    text: str


class EpisodicMemory:
    """情景记忆：跨尝试累积的反思列表。"""

    def __init__(self) -> None:
        self._items: list[Reflection] = []

    def add(self, reflection: Reflection) -> None:
        """
        Args:
            reflection: 新反思。
        """
        self._items.append(reflection)

    def as_prompt_block(self, max_items: int = 5) -> str:
        """
        将近期反思格式化为下次 Actor 提示前缀。

        Args:
            max_items: 最多保留条数（防记忆腐烂示意）。

        Returns:
            block: 可注入提示词的文本；无记忆时为空串。
        """
        if not self._items:
            return ""
        recent = self._items[-max_items:]
        lines = ["# Episodic reflections (do not repeat past mistakes)"]
        for r in recent:
            lines.append(f"- [attempt {r.attempt}] {r.text}")
        return "\n".join(lines)

    def __len__(self) -> int:
        return len(self._items)

    @property
    def items(self) -> list[Reflection]:
        """
        Returns:
            items: 反思副本列表。
        """
        return list(self._items)


class ToolRegistry:
    """按名分发工具。"""

    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}

    def register(self, name: str, fn: Callable[..., str]) -> None:
        """
        Args:
            name: 工具名。
            fn: 返回 ``str`` 的函数。
        """
        self._tools[name] = fn

    def dispatch(self, call: ToolCall) -> str:
        """
        Args:
            call: 工具调用。

        Returns:
            observation: 结果或 ``Error: ...``。
        """
        fn = self._tools.get(call.name)
        if fn is None:
            return f"Error: unknown tool {call.name}"
        try:
            return fn(**call.args)
        except Exception as e:
            return f"Error: {type(e).__name__}: {e}"


def calculator(expr: str) -> str:
    """
    Args:
        expr: 算术表达式。

    Returns:
        value: 计算结果或错误。
    """
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "Error: illegal characters"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))  # noqa: S307
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


class ActorScript(TypedDict, total=False):
    """脚本化 Actor 一步。"""

    thought: str
    action: str
    args: dict[str, Any]
    final: str


class ScriptedActor:
    """
    玩具 Actor：按剧本跑“一次尝试”。

    真实系统会把 ``memory.as_prompt_block()`` 拼进 system prompt；
    这里用 ``memory_aware`` 剧本分支模拟“看见反思后改行为”。
    """

    def __init__(
        self,
        tools: ToolRegistry,
        script_blind: list[ActorScript],
        script_aware: list[ActorScript] | None = None,
    ) -> None:
        self.tools = tools
        self.script_blind = script_blind
        self.script_aware = script_aware or script_blind

    def run(self, question: str, memory: EpisodicMemory) -> Trajectory:
        """
        Args:
            question: 用户问题。
            memory: 情景记忆（非空则用 aware 剧本）。

        Returns:
            traj: 本次尝试轨迹。
        """
        script = self.script_aware if len(memory) > 0 else self.script_blind
        traj = Trajectory(question=question)
        for entry in script:
            if "final" in entry:
                traj.final_answer = str(entry["final"])
                break
            call = ToolCall(str(entry["action"]), dict(entry.get("args") or {}))
            obs = self.tools.dispatch(call)
            traj.steps.append(
                Step(
                    thought=str(entry.get("thought", "")),
                    action=call.name,
                    args=call.args,
                    observation=obs,
                )
            )
        return traj


def evaluate_scalar(traj: Trajectory, expected_answer: str) -> EvalResult:
    """
    标量评估：最终答案是否精确匹配（最强外部信号）。

    Args:
        traj: 轨迹。
        expected_answer: 标准答案。

    Returns:
        result: 二分成功 + score∈{0,1}。
    """
    ok = traj.final_answer.strip() == expected_answer.strip()
    return EvalResult(
        kind=EvalKind.SCALAR,
        success=ok,
        score=1.0 if ok else 0.0,
        reason="exact match" if ok else f"got={traj.final_answer!r} expected={expected_answer!r}",
    )


def evaluate_heuristic(traj: Trajectory, max_steps: int = 4) -> EvalResult:
    """
    启发式评估：重复行动 / 过长轨迹 / 工具错误。

    Args:
        traj: 轨迹。
        max_steps: 步数上限。

    Returns:
        result: 失败签名检测。
    """
    actions = [(s.action, str(s.args)) for s in traj.steps]
    if len(actions) >= 2 and actions[-1] == actions[-2]:
        return EvalResult(
            EvalKind.HEURISTIC, False, 0.0, "repeated identical action (stuck)"
        )
    if len(traj.steps) > max_steps:
        return EvalResult(
            EvalKind.HEURISTIC, False, 0.2, f"trajectory longer than {max_steps} steps"
        )
    if any(s.observation.startswith("Error:") for s in traj.steps):
        return EvalResult(EvalKind.HEURISTIC, False, 0.1, "tool error in trajectory")
    return EvalResult(EvalKind.HEURISTIC, True, 0.8, "no failure signature")


def evaluate_self_toy(traj: Trajectory) -> EvalResult:
    """
    玩具自我评估：有最终答案且最后观察非 Error 则偏成功（弱信号）。

    Args:
        traj: 轨迹。

    Returns:
        result: 弱打分。
    """
    if not traj.final_answer:
        return EvalResult(EvalKind.SELF, False, 0.0, "missing final answer")
    if traj.steps and traj.steps[-1].observation.startswith("Error:"):
        return EvalResult(EvalKind.SELF, False, 0.2, "last observation is error")
    return EvalResult(EvalKind.SELF, True, 0.6, "self-eval: looks plausible")


def hybrid_evaluate(
    traj: Trajectory,
    expected_answer: str | None = None,
) -> EvalResult:
    """
    2026 默认混合：有标量用标量，否则启发式，再退回自我评估。

    Args:
        traj: 轨迹。
        expected_answer: 可选标准答案。

    Returns:
        result: 选用的评估结果。
    """
    if expected_answer is not None:
        scalar = evaluate_scalar(traj, expected_answer)
        if not scalar.success:
            # 仍跑启发式补充 reason
            h = evaluate_heuristic(traj)
            if not h.success:
                return EvalResult(
                    EvalKind.SCALAR,
                    False,
                    0.0,
                    f"{scalar.reason}; also heuristic: {h.reason}",
                )
        return scalar
    h = evaluate_heuristic(traj)
    if not h.success:
        return h
    return evaluate_self_toy(traj)


def reflect_toy(traj: Trajectory, evaluation: EvalResult, attempt: int) -> Reflection:
    """
    玩具 Self-Reflector：根据失败原因写自然语言反思。

    Args:
        traj: 失败轨迹。
        evaluation: 评估结果。
        attempt: 尝试编号（从 1 起）。

    Returns:
        reflection: 写入情景记忆的反思。
    """
    used = [s.action for s in traj.steps]
    text = (
        f"Attempt {attempt} failed ({evaluation.kind.value}): {evaluation.reason}. "
        f"Tools used: {used or ['<none>']}. "
        f"Next time: verify the question intent before calling tools; "
        f"prefer calculator for arithmetic; avoid repeating the same failing call."
    )
    return Reflection(attempt=attempt, text=text)


@dataclass
class ReflexionLoop:
    """Actor / Evaluator / Reflector + 情景记忆的重试循环。"""

    actor: ScriptedActor
    max_attempts: int = 3
    memory: EpisodicMemory = field(default_factory=EpisodicMemory)

    def run(
        self,
        question: str,
        expected_answer: str | None = None,
    ) -> tuple[Trajectory, EvalResult, EpisodicMemory]:
        """
        Args:
            question: 用户问题。
            expected_answer: 可选标量答案。

        Returns:
            traj: 最后一次轨迹。
            evaluation: 最后一次评估。
            memory: 累积情景记忆。
        """
        last_traj = Trajectory(question=question)
        last_eval = EvalResult(EvalKind.SELF, False, 0.0, "not run")
        for attempt in range(1, self.max_attempts + 1):
            last_traj = self.actor.run(question, self.memory)
            last_eval = hybrid_evaluate(last_traj, expected_answer)
            if last_eval.success:
                return last_traj, last_eval, self.memory
            self.memory.add(reflect_toy(last_traj, last_eval, attempt))
        return last_traj, last_eval, self.memory


print("Reflexion toy ready | actor/evaluator/reflector/memory")


## 2. 玩具示例：先失败再靠反思改行为


In [ ]:
def demo_reflexion_retry() -> None:
    """盲试误用错误工具 → 反思 → 第二次用 calculator 成功。"""
    tools = ToolRegistry()
    tools.register("calculator", calculator)
    tools.register(
        "search_kb",
        lambda query: "Error: kb offline",
    )

    # 第一次：错误地去搜知识库
    blind: list[ActorScript] = [
        {
            "thought": "去知识库找答案",
            "action": "search_kb",
            "args": {"query": "(3+5)*7"},
        },
        {"final": "不知道"},
    ]
    # 看见反思后：改用计算器
    aware: list[ActorScript] = [
        {
            "thought": "反思提醒：算术应用 calculator",
            "action": "calculator",
            "args": {"expr": "(3+5)*7"},
        },
        {"final": "56"},
    ]
    loop = ReflexionLoop(
        actor=ScriptedActor(tools, blind, aware),
        max_attempts=3,
    )
    traj, ev, mem = loop.run("计算 (3+5)*7", expected_answer="56")
    print("=== final eval ===", ev)
    print("=== final answer ===", traj.final_answer)
    print("=== episodic memory ===")
    print(mem.as_prompt_block())
    assert ev.success and traj.final_answer == "56"
    assert len(mem) >= 1


def demo_heuristic_stuck() -> None:
    """启发式捕获重复相同行动。"""
    traj = Trajectory(
        question="x",
        steps=[
            Step("t", "calculator", {"expr": "1+1"}, "2"),
            Step("t", "calculator", {"expr": "1+1"}, "2"),
        ],
        final_answer="2",
    )
    h = evaluate_heuristic(traj)
    print("\n=== heuristic stuck ===", h)
    assert not h.success and "repeated" in h.reason


demo_reflexion_retry()
demo_heuristic_stuck()
print("\nTOY DEMOS OK")


## 3. PyTorch：可学习轨迹评估器（自我评估蒸馏占位）

用轨迹特征（步数、错误次数、是否有最终答案等）训练一个小 MLP，输出成功概率——对应笔记里可替换的 Evaluator。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def trajectory_features(traj: Trajectory) -> torch.Tensor:
    """
    Args:
        traj: 轨迹。

    Returns:
        x: ``(5,)`` 手工特征。
    """
    n = float(len(traj.steps))
    n_err = float(sum(1 for s in traj.steps if s.observation.startswith("Error:")))
    has_final = 1.0 if traj.final_answer.strip() else 0.0
    uniq_actions = float(len({s.action for s in traj.steps}))
    repeat = 0.0
    if len(traj.steps) >= 2:
        a = (traj.steps[-1].action, str(traj.steps[-1].args))
        b = (traj.steps[-2].action, str(traj.steps[-2].args))
        repeat = 1.0 if a == b else 0.0
    return torch.tensor([n, n_err, has_final, uniq_actions, repeat], dtype=torch.float32)


class TrajectoryEvaluatorNet(nn.Module):
    """轨迹 → 成功 logits（教学规模）。"""

    def __init__(self, in_dim: int = 5) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, 5)`` 或 ``(5,)``。

        Returns:
            logit: ``(B,)`` 或标量 logit。
        """
        single = x.ndim == 1
        if single:
            x = x.unsqueeze(0)
        logit = self.net(x).squeeze(-1)
        return logit.squeeze(0) if single else logit

    @torch.no_grad()
    def score(self, traj: Trajectory) -> EvalResult:
        """
        Args:
            traj: 轨迹。

        Returns:
            result: ``kind=SELF``，``score=sigmoid(logit)``。
        """
        logit = self.forward(trajectory_features(traj))
        p = float(torch.sigmoid(logit).item())
        ok = p >= 0.5
        return EvalResult(
            kind=EvalKind.SELF,
            success=ok,
            score=p,
            reason=f"nn self-eval p={p:.3f}",
        )


def train_trajectory_evaluator(steps: int = 300, lr: float = 0.05) -> TrajectoryEvaluatorNet:
    """
    在合成轨迹特征上 BCE 训练。

    Args:
        steps: 优化步数。
        lr: 学习率。

    Returns:
        model: 训练后的评估器。
    """
    # (features-like Trajectory stubs via manual tensors, labels)
    pos = [
        Trajectory("q", [Step("t", "calculator", {"expr": "1+1"}, "2")], "2"),
        Trajectory("q", [Step("t", "calculator", {"expr": "3*3"}, "9")], "9"),
    ]
    neg = [
        Trajectory("q", [Step("t", "search_kb", {"query": "x"}, "Error: fail")], ""),
        Trajectory(
            "q",
            [
                Step("t", "calculator", {"expr": "1"}, "1"),
                Step("t", "calculator", {"expr": "1"}, "1"),
            ],
            "1",
        ),
    ]
    model = TrajectoryEvaluatorNet()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        loss = torch.tensor(0.0)
        for traj, y in [(t, 1.0) for t in pos] + [(t, 0.0) for t in neg]:
            logit = model(trajectory_features(traj))
            target = torch.tensor(y)
            loss = loss + F.binary_cross_entropy_with_logits(logit, target)
        loss = loss / (len(pos) + len(neg))
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


def demo_nn_evaluator() -> None:
    """训练并对比好坏轨迹分数。"""
    torch.manual_seed(0)
    model = train_trajectory_evaluator()
    good = Trajectory("q", [Step("t", "calculator", {"expr": "2+2"}, "4")], "4")
    bad = Trajectory("q", [Step("t", "search_kb", {"query": "2+2"}, "Error: x")], "")
    g, b = model.score(good), model.score(bad)
    print("=== nn evaluator ===")
    print("good:", g)
    print("bad:", b)
    assert g.score > b.score
    print("NN EVAL OK")


demo_nn_evaluator()


## 4. 生产级：LangGraph Reflexion + DeepSeek

图结构：`actor → evaluate → (成功结束 | reflect → actor)`。  
情景记忆写入 state，下一轮 Actor 的 system 前缀带上反思。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"


@tool
def calculator_tool(expr: str) -> str:
    """Evaluate a simple arithmetic expression like '(3+5)*7'."""
    return calculator(expr)


@tool
def search_kb_tool(query: str) -> str:
    """Search a tiny offline KB. Prefer calculator_tool for math."""
    q = query.lower()
    if "france" in q and "capital" in q:
        return "Paris"
    return f"Error: missing kb entry for {query}"


class Grade(BaseModel):
    """评估结构化输出。"""

    success: bool = Field(description="whether the attempt solved the user question")
    score: float = Field(ge=0.0, le=1.0, description="quality score")
    reason: str = Field(description="brief reason")


class ReflectionOut(BaseModel):
    """反思结构化输出。"""

    reflection: str = Field(description="natural-language lesson for the next attempt")


class ReflexionState(TypedDict, total=False):
    """LangGraph 状态。"""

    question: str
    expected_answer: str
    messages: list
    reflections: list[str]
    attempt: int
    max_attempts: int
    success: bool
    score: float
    eval_reason: str
    final_answer: str


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def _memory_block(reflections: list[str]) -> str:
    """
    Args:
        reflections: 已有反思。

    Returns:
        block: system 前缀片段。
    """
    if not reflections:
        return ""
    lines = ["Episodic reflections from prior failed attempts:"]
    for i, r in enumerate(reflections, 1):
        lines.append(f"{i}. {r}")
    lines.append("Do not repeat those mistakes.")
    return "\n".join(lines)


def actor_node(state: ReflexionState) -> dict[str, Any]:
    """
    单次尝试：带工具的 Actor 循环（有限步 tool-calling）。

    Args:
        state: 含 question / reflections。

    Returns:
        update: ``messages`` / ``attempt`` / ``final_answer``。
    """
    llm = get_llm().bind_tools([calculator_tool, search_kb_tool])
    tools_by_name = {
        calculator_tool.name: calculator_tool,
        search_kb_tool.name: search_kb_tool,
    }
    mem = _memory_block(list(state.get("reflections") or []))
    system = (
        "You are the Actor in a Reflexion agent. "
        "Solve the user question using tools when needed. "
        "For arithmetic always use calculator_tool. "
        "Finish with a short final answer in Chinese."
    )
    if mem:
        system = system + "\n\n" + mem

    messages: list[BaseMessage] = [
        SystemMessage(content=system),
        HumanMessage(content=state["question"]),
    ]
    # 有限 tool 轮次，避免无限循环
    for _ in range(6):
        ai: AIMessage = llm.invoke(messages)  # type: ignore[assignment]
        messages.append(ai)
        if not ai.tool_calls:
            break
        for tc in ai.tool_calls:
            name = tc["name"]
            args = tc.get("args") or {}
            fn = tools_by_name.get(name)
            if fn is None:
                obs = f"Error: unknown tool {name}"
            else:
                obs = str(fn.invoke(args))
            messages.append(
                ToolMessage(content=obs, tool_call_id=tc["id"], name=name)
            )

    final = ""
    for m in reversed(messages):
        if isinstance(m, AIMessage) and m.content and not m.tool_calls:
            final = m.content if isinstance(m.content, str) else str(m.content)
            break
    return {
        "messages": messages,
        "attempt": int(state.get("attempt", 0)) + 1,
        "final_answer": final,
    }


def evaluate_node(state: ReflexionState) -> dict[str, Any]:
    """
    混合评估：若提供 expected_answer 则标量优先，否则 LLM 自我评估。

    Args:
        state: 当前尝试状态。

    Returns:
        update: success / score / eval_reason。
    """
    expected = (state.get("expected_answer") or "").strip()
    final = (state.get("final_answer") or "").strip()
    if expected:
        # 标量：允许答案文本包含期望子串（中文句中嵌入数字）
        ok = expected == final or expected in final
        return {
            "success": ok,
            "score": 1.0 if ok else 0.0,
            "eval_reason": "scalar match" if ok else f"scalar miss: {final!r}",
        }

    grader = get_llm().with_structured_output(Grade)
    grade: Grade = grader.invoke(
        [
            SystemMessage(
                content=(
                    "You are the Evaluator. Grade whether the actor solved the question. "
                    "Be strict."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "question": state["question"],
                        "final_answer": final,
                        "messages_tail": [
                            getattr(m, "content", str(m)) for m in (state.get("messages") or [])[-6:]
                        ],
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )
    return {
        "success": bool(grade.success),
        "score": float(grade.score),
        "eval_reason": grade.reason,
    }


def reflect_node(state: ReflexionState) -> dict[str, Any]:
    """
    Self-Reflector：写反思并追加到情景记忆。

    Args:
        state: 失败尝试状态。

    Returns:
        update: ``reflections``。
    """
    reflector = get_llm().with_structured_output(ReflectionOut)
    out: ReflectionOut = reflector.invoke(
        [
            SystemMessage(
                content=(
                    "You are the Self-Reflector. The actor failed. "
                    "Write one concise reflection that will help the NEXT attempt. "
                    "Focus on concrete mistakes (wrong tool, misread question)."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "question": state["question"],
                        "final_answer": state.get("final_answer"),
                        "eval_reason": state.get("eval_reason"),
                        "attempt": state.get("attempt"),
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )
    reflections = list(state.get("reflections") or [])
    reflections.append(out.reflection)
    return {"reflections": reflections}


def route_after_eval(state: ReflexionState) -> Literal["end", "reflect", "end_fail"]:
    """
    Args:
        state: 评估后状态。

    Returns:
        next: 成功结束 / 反思 / 耗尽失败。
    """
    if state.get("success"):
        return "end"
    if int(state.get("attempt", 0)) >= int(state.get("max_attempts", 3)):
        return "end_fail"
    return "reflect"


def build_reflexion_graph() -> Any:
    """
    Returns:
        app: 编译后的 Reflexion 图。
    """
    g: StateGraph = StateGraph(ReflexionState)
    g.add_node("actor", actor_node)
    g.add_node("evaluate", evaluate_node)
    g.add_node("reflect", reflect_node)
    g.add_edge(START, "actor")
    g.add_edge("actor", "evaluate")
    g.add_conditional_edges(
        "evaluate",
        route_after_eval,
        {"end": END, "end_fail": END, "reflect": "reflect"},
    )
    g.add_edge("reflect", "actor")
    return g.compile()


def run_reflexion(
    question: str,
    *,
    expected_answer: str = "",
    max_attempts: int = 3,
) -> ReflexionState:
    """
    Args:
        question: 用户问题。
        expected_answer: 可选标量答案（如 ``\"56\"``）。
        max_attempts: 最大尝试次数。

    Returns:
        state: 最终图状态。
    """
    app = build_reflexion_graph()
    return app.invoke(
        {
            "question": question,
            "expected_answer": expected_answer,
            "messages": [],
            "reflections": [],
            "attempt": 0,
            "max_attempts": max_attempts,
            "success": False,
            "score": 0.0,
            "eval_reason": "",
            "final_answer": "",
        }
    )


print("LangGraph Reflexion ready |", MODEL)


## 5. 生产示例


In [ ]:
def demo_deepseek_reflexion() -> None:
    """需要网络与 ``DEEPSEEK_API_KEY``。"""
    out = run_reflexion(
        "请计算 (3+5)*7，只把数字结果作为最终答案的一部分。",
        expected_answer="56",
        max_attempts=3,
    )
    print("=== attempts ===", out.get("attempt"))
    print("=== success ===", out.get("success"), "score=", out.get("score"))
    print("=== eval_reason ===", out.get("eval_reason"))
    print("=== reflections ===")
    for i, r in enumerate(out.get("reflections") or [], 1):
        print(f"{i}. {r}")
    print("=== final_answer ===")
    print(out.get("final_answer"))
    assert out.get("success"), "should solve with scalar signal"


demo_deepseek_reflexion()
